In [ ]:
# NOTE: Run this in a Colab environment with GPU hardware acceleration enabled
!pip install ultralytics

from ultralytics import YOLO

# load pretrained YOLOv8n model
model = YOLO("yolov8n-pose.pt")  

# fine-tune for hand tracking
model.train(data="hand-keypoints.yaml", epochs=100, imgsz=320, device=0, batch=-1)

In [ ]:
# add an environmental flag to force pure CPU routing for the toolkit export
import os
os.environ["CUDA_VISIBLE_DEVICES"] = ""

# load hand model & prepare for export
from ultralytics import YOLO
model = YOLO("./yolov8n-hand.pt")

In [ ]:
# Select calibration images from downloaded hand keypoint images for exporting, since imx requires quantization 
from pathlib import Path
import random
import shutil

# ----------------------------
SRC_IMG_DIR = Path("/content/datasets/hand-keypoints/images/val")
SRC_LBL_DIR = Path("/content/datasets/hand-keypoints/labels/val")

DST_IMG_DIR = Path("/content/datasets/hand-keypoints/images/calib")
DST_LBL_DIR = Path("/content/datasets/hand-keypoints/labels/calib")

NUM_IMAGES = 100
SEED = 42

DST_IMG_DIR.mkdir(parents=True, exist_ok=True)
DST_LBL_DIR.mkdir(parents=True, exist_ok=True)

valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

images = [p for p in SRC_IMG_DIR.iterdir() if p.suffix.lower() in valid_exts]

random.seed(SEED)
selected = random.sample(images, NUM_IMAGES)

missing_labels = 0

for i, img_path in enumerate(selected):
    # copy image
    dst_img = DST_IMG_DIR / f"{i:05d}{img_path.suffix.lower()}"
    shutil.copy2(img_path, dst_img)

    # copy matching label
    label_path = SRC_LBL_DIR / (img_path.stem + ".txt")

    if label_path.exists():
        dst_lbl = DST_LBL_DIR / f"{i:05d}.txt"
        shutil.copy2(label_path, dst_lbl)
    else:
        missing_labels += 1

print(f"Copied {NUM_IMAGES} images")
print(f"Missing labels: {missing_labels}")

In [ ]:

# export to imx format!
model.export(data="./hand-keypoints-calib.yaml", format="imx", imgsz=320, device="cpu", optimize=False, batch=1)

In [ ]:
!zip -r model.zip /content/best_imx_model/
from google.colab import files

In [ ]:
files.download("model.zip")